In [1]:
!pip install transformers datasets seqeval -q

import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification 
)
from seqeval.metrics import precision_score, recall_score, f1_score

W0403 19:28:42.579000 5824 anaconda3\Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [2]:
dataset = load_dataset("wnut_17", trust_remote_code=True)

label_list = dataset["train"].features["ner_tags"].feature.names
num_labels = len(label_list)

print(label_list)

['O', 'B-corporation', 'I-corporation', 'B-creative-work', 'I-creative-work', 'B-group', 'I-group', 'B-location', 'I-location', 'B-person', 'I-person', 'B-product', 'I-product']


In [3]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_and_align_labels(examples):
    tokenized = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        previous_word = None
        label_ids = []

        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            elif word_id != previous_word:
                label_ids.append(label[word_id])
            else:
                label_ids.append(-100)
            previous_word = word_id

        labels.append(label_ids)

    tokenized["labels"] = labels
    return tokenized

tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=True)

C:\Users\Tanisha\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Map:   0%|          | 0/1287 [00:00<?, ? examples/s]

In [7]:
data_collator = DataCollatorForTokenClassification(tokenizer)

In [9]:
model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_labels
)

Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [11]:
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_labels = [[label_list[l] for l in label if l != -100] for label in labels]
    true_preds = [
        [label_list[p] for (p, l) in zip(pred, label) if l != -100]
        for pred, label in zip(predictions, labels)
    ]

    return {
        "precision": precision_score(true_labels, true_preds),
        "recall": recall_score(true_labels, true_preds),
        "f1": f1_score(true_labels, true_preds),
    }

In [13]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    evaluation_strategy="epoch",
    save_strategy="no"
)

In [15]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"].select(range(1000)),
    eval_dataset=tokenized_dataset["validation"].select(range(300)),
    tokenizer=tokenizer,
    data_collator=data_collator, 
    compute_metrics=compute_metrics
)

In [17]:
trainer.train()

C:\Users\Tanisha\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,No log,0.419735,0.000000,0.000000,0.000000


C:\Users\Tanisha\anaconda3\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


TrainOutput(global_step=125, training_loss=0.40645050048828124, metrics={'train_runtime': 433.2688, 'train_samples_per_second': 2.308, 'train_steps_per_second': 0.289, 'total_flos': 12095982905280.0, 'train_loss': 0.40645050048828124, 'epoch': 1.0})

In [19]:
results = trainer.evaluate()
print(results)

{'eval_loss': 0.4197346866130829, 'eval_precision': 0.0, 'eval_recall': 0.0, 'eval_f1': 0.0, 'eval_runtime': 21.7023, 'eval_samples_per_second': 13.823, 'eval_steps_per_second': 1.751, 'epoch': 1.0}


In [21]:
def predict(sentence):
    tokens = tokenizer(sentence.split(), return_tensors="pt", is_split_into_words=True)

    outputs = model(**tokens)
    preds = np.argmax(outputs.logits.detach().numpy(), axis=2)[0]

    words = sentence.split()
    tags = [label_list[p] for p in preds[:len(words)]]

    for w, t in zip(words, tags):
        print(w, "->", t)

predict("John works at Google in California")

John -> O
works -> O
at -> O
Google -> O
in -> O
California -> O


Dataset: WNUT-17

This task demonstrates token classification using DistilBERT.
It can be applied to POS tagging and chunking tasks.

Label alignment is handled properly to deal with subword tokenization.
